# EDA A2 – Kim są klienci?

Pytanie biznesowe: **kim są nasi klienci i które segmenty są najbardziej wartościowe?**

W A1 okazało się, że kanały pozyskania prawie się nie różnią wartością klienta.
Szukamy więc wymiarów, które się różnią – wiek, płeć, kraj – bo na nich zbudujemy rekomendacje
(np. do kogo kierować reklamy).

Zasady z A1:
* przychód = suma `order_items.sale_price` dla statusów **Complete + Shipped**,
* analizy w czasie opieramy na **`orders.created_at`** (`order_items.created_at` jest niewiarygodne),
* zawsze zapisujemy, na jaki moment są dane.

## Wczytanie danych

* `users` – profil klienta (bez danych osobowych).
* `orders` – status i data zamówienia.
* `order_items` – tylko `order_id` i `sale_price`: wystarczy, żeby policzyć wartość zamówienia.

In [1]:
import pandas as pd
import numpy as np

from bq_utils import read_bq

REVENUE_STATUSES = ["Complete", "Shipped"]

c:\Users\sobie\Desktop\Python_Workspace\thelook-ecommerce-analytics\.venv\Lib\site-packages\google\auth\_default.py:113: UserWarning: Your application has authenticated using end user credentials from Google Cloud SDK without a quota project. You might receive a "quota exceeded" or "API not enabled" error. See the following page for troubleshooting: https://cloud.google.com/docs/authentication/adc-troubleshooting/user-creds. 
  warnings.warn(_CLOUD_SDK_CREDENTIALS_WARNING)


In [2]:
# Users

users_cols = ["id", "age", "gender", "country", "traffic_source", "created_at"]

users_dtypes = {"id": "Int32", "age": "Int8", "gender": "category",
                "country": "category", "traffic_source": "category"}

users = read_bq("users", usecols=users_cols, dtype=users_dtypes)

users.head()

Zapytanie przetworzy: 4.52 MB


c:\Users\sobie\Desktop\Python_Workspace\thelook-ecommerce-analytics\.venv\Lib\site-packages\google\cloud\bigquery\table.py:2122: UserWarning: BigQuery Storage module not found, fetch data with the REST endpoint instead.
  warnings.warn(


,id,age,gender,country,traffic_source,created_at
0,18258,12,M,Brasil,Email,2026-05-25 16:27:00+00:00
1,75380,12,F,Brasil,Search,2019-11-29 12:39:00+00:00
2,4820,12,F,Japan,Email,2023-10-03 00:30:00+00:00
3,6415,12,F,Brasil,Search,2019-09-19 05:36:00+00:00
4,70422,12,F,Brasil,Display,2024-01-20 15:08:00+00:00


In [3]:
# Orders

orders_cols = ["order_id", "user_id", "status", "created_at"]

orders_dtypes = {"order_id": "Int32", "user_id": "Int32", "status": "category"}

orders = read_bq("orders", usecols=orders_cols, dtype=orders_dtypes)

orders.head()

Zapytanie przetworzy: 4.26 MB


c:\Users\sobie\Desktop\Python_Workspace\thelook-ecommerce-analytics\.venv\Lib\site-packages\google\cloud\bigquery\table.py:2122: UserWarning: BigQuery Storage module not found, fetch data with the REST endpoint instead.
  warnings.warn(


,order_id,user_id,status,created_at
0,6,4,Cancelled,2026-03-28 20:13:14+00:00
1,8,5,Cancelled,2024-08-13 18:25:40+00:00
2,38,28,Cancelled,2026-02-03 03:46:46+00:00
3,52,37,Cancelled,2024-07-08 20:05:34+00:00
4,55,41,Cancelled,2026-09-02 11:34:36+00:00


In [4]:
# Order items – tylko to, co potrzebne do wartości zamówienia

order_items = read_bq("order_items", usecols=["order_id", "sale_price"],
                      dtype={"order_id": "Int32"})

order_items.head()

Zapytanie przetworzy: 2.89 MB


c:\Users\sobie\Desktop\Python_Workspace\thelook-ecommerce-analytics\.venv\Lib\site-packages\google\cloud\bigquery\table.py:2122: UserWarning: BigQuery Storage module not found, fetch data with the REST endpoint instead.
  warnings.warn(


,order_id,sale_price
0,2526,0.02
1,18153,0.02
2,47981,0.02
3,84256,0.02
4,348,0.02


In [5]:
# Stan danych – zapisujemy moment pobrania (zbiór zmienia się codziennie)

checked_at = pd.Timestamp.now(tz="UTC")
print(f"Sprawdzono: {checked_at:%Y-%m-%d %H:%M} UTC")
print(f"Najnowszy użytkownik: {users['created_at'].max():%Y-%m-%d %H:%M}")
print(f"Najnowsze zamówienie: {orders['created_at'].max():%Y-%m-%d %H:%M}")

Sprawdzono: 2026-10-29 08:21 UTC
Najnowszy użytkownik: 2026-10-28 19:27
Najnowsze zamówienie: 2026-10-29 00:45


## 1. Profil klientów

* Jaki jest rozkład płci? Jakie są najczęstsze kraje (top 10 + udział w %)?
* Jaki jest rozkład wieku? Opisz go `describe()`, a potem podziel na przedziały
  `pd.cut` i dodaj jako kolumnę `age_group`.

In [6]:
# Podział wg płci w % – normalize=True zwraca udziały zamiast liczebności

users["gender"].value_counts(normalize=True).mul(100).round(1)

gender
M    50.2
F    49.8
Name: proportion, dtype: float64

In [7]:
# Top 10 krajów – liczba i udział użytkowników

countries = (users["country"]
             .value_counts()
             .to_frame("users")
             .assign(share_pct=lambda x: (x["users"] / x["users"].sum() * 100).round(1))
             .head(10))

countries

,users,share_pct
country,,
China,33738,33.7
United States,22750,22.8
Brasil,14576,14.6
South Korea,5384,5.4
France,4701,4.7
United Kingdom,4510,4.5
Germany,4198,4.2
Spain,4021,4.0
Japan,2438,2.4


In [8]:
# Rozkład wieku

users["age"].describe().round(1)

count    100000.0
mean         41.0
std          17.0
min          12.0
25%          26.0
50%          41.0
75%          56.0
max          70.0
Name: age, dtype: Float64

In [9]:
# Grupy wieku – pd.cut przypisuje wiek do przedziału, prawy koniec domknięty: (17, 24] = 18–24

age_bins = [0, 17, 24, 34, 44, 54, 64, 120]
age_labels = ["12-17", "18-24", "25-34", "35-44", "45-54", "55-64", "65+"]

users["age_group"] = pd.cut(users["age"].astype("float"), bins=age_bins, labels=age_labels)

users["age_group"].value_counts(normalize=True, sort=False).mul(100).round(1)

age_group
12-17    10.1
18-24    12.0
25-34    16.9
35-44    16.9
45-54    16.9
55-64    16.9
65+      10.3
Name: proportion, dtype: float64

## 2. Wzrost bazy klientów

* Ilu nowych użytkowników rejestruje się co miesiąc?
* Czy liczba rośnie, czy jest stała? Pomijamy bieżący, niepełny miesiąc.

In [10]:
# Nowi użytkownicy miesięcznie – bez strefy czasowej, data -> miesiąc (Period), zliczenie

signup_month = users["created_at"].dt.tz_localize(None).dt.to_period("M")
current_month = checked_at.tz_localize(None).to_period("M")

new_users = signup_month.value_counts().sort_index()
new_users = new_users[new_users.index < current_month]      # tylko pełne miesiące

new_users.tail(12)

created_at
2025-10    1103
2025-11    1095
2025-12    1078
2026-01    1049
2026-02     913
2026-03    1027
2026-04    1007
2026-05    1072
2026-06    1018
2026-07    1043
2026-08    1067
2026-09    1019
Freq: M, Name: count, dtype: int64

In [11]:
# Średnio nowych użytkowników na miesiąc w każdym roku – czy baza rośnie?

new_users.groupby(new_users.index.year).mean().round(0)

created_at
2019    1022.0
2020    1054.0
2021    1038.0
2022    1037.0
2023    1022.0
2024    1065.0
2025    1043.0
2026    1024.0
Name: count, dtype: float64

## 3. Wartość segmentów

Najpierw budujemy tabelę **jeden wiersz = jeden użytkownik** z kolumnami `orders` i `revenue`
(tylko zamówienia Complete + Shipped):

1. Wartość każdego zamówienia z `order_items` (`groupby("order_id")`).
2. Dołączamy ją do `orders` i zostawiamy statusy z `REVENUE_STATUSES`.
3. Sumujemy per `user_id` i dołączamy do `users` przez **left join** – użytkownicy bez zakupów
   zostają z zerami (`.fillna(0)` to odpowiednik `COALESCE(..., 0)` z SQL).

Potem dla `age_group`, `gender` i `country` (top 10) liczymy: users, buyer rate, AOV, revenue per user.
Czy któryś wymiar różni się wyraźniej niż kanały pozyskania w A1?

In [12]:
# Tabela na użytkownika: zamówienia i przychód (Complete + Shipped), bez zakupów = 0

order_value = order_items.groupby("order_id")["sale_price"].sum().rename("order_value")

user_stats = (orders[orders["status"].isin(REVENUE_STATUSES)]
              .merge(order_value, left_on="order_id", right_index=True, how="left")
              .groupby("user_id")
              .agg(orders=("order_id", "count"), revenue=("order_value", "sum")))

customers = (users
             .merge(user_stats, left_on="id", right_index=True, how="left")
             .fillna({"orders": 0, "revenue": 0}))

# Kontrola: złączenie nie gubi ani nie dubluje użytkowników, suma przychodu bez zmian
print(f"Users: {len(users):,} -> customers: {len(customers):,}")
print(f"Revenue: {user_stats['revenue'].sum():,.0f} -> {customers['revenue'].sum():,.0f}")

customers.head()

Users: 100,000 -> customers: 100,000
Revenue: 5,952,870 -> 5,952,870


,id,age,gender,country,traffic_source,created_at,age_group,orders,revenue
0,18258,12,M,Brasil,Email,2026-05-25 16:27:00+00:00,12-17,1,25.00
1,75380,12,F,Brasil,Search,2019-11-29 12:39:00+00:00,12-17,1,46.00
2,4820,12,F,Japan,Email,2023-10-03 00:30:00+00:00,12-17,1,7.97
3,6415,12,F,Brasil,Search,2019-09-19 05:36:00+00:00,12-17,0,0.00
4,70422,12,F,Brasil,Display,2024-01-20 15:08:00+00:00,12-17,0,0.00


In [13]:
# KPI wg segmentu – jedna funkcja dla każdego wymiaru

def segment_kpi(df, by):
    return (df.groupby(by, observed=True)
              .agg(users=("id", "count"),
                   buyers=("orders", lambda s: (s > 0).sum()),
                   orders=("orders", "sum"),
                   revenue=("revenue", "sum"))
              .assign(buyer_rate_pct=lambda x: (x["buyers"] / x["users"] * 100).round(1),
                      aov=lambda x: (x["revenue"] / x["orders"]).round(2),
                      revenue_per_user=lambda x: (x["revenue"] / x["users"]).round(2))
              .drop(columns=["buyers", "orders"])
              .round({"revenue": 0}))


segment_kpi(customers, "age_group")

,users,revenue,buyer_rate_pct,aov,revenue_per_user
age_group,,,,,
12-17,10117,602629.0,52.3,87.2,59.57
18-24,11981,738818.0,53.0,89.28,61.67
25-34,16851,995060.0,53.0,86.29,59.05
35-44,16909,1014181.0,52.8,87.02,59.98
45-54,16908,995231.0,52.3,86.74,58.86
55-64,16939,978992.0,52.2,85.01,57.8
65+,10295,627959.0,54.2,85.97,61.0


In [14]:
segment_kpi(customers, "gender")

,users,revenue,buyer_rate_pct,aov,revenue_per_user
gender,,,,,
F,49805,2781425.0,52.7,81.27,55.85
M,50195,3171445.0,52.9,92.08,63.18


In [15]:
# Tylko top 10 krajów – małe kraje dają zaszumione średnie

top_countries = customers[customers["country"].isin(countries.index)]

segment_kpi(top_countries, "country").sort_values("revenue_per_user", ascending=False)

,users,revenue,buyer_rate_pct,aov,revenue_per_user
country,,,,,
Japan,2438,157720.0,53.9,89.21,64.69
Germany,4198,260144.0,53.8,88.24,61.97
China,33738,2011625.0,52.9,86.7,59.62
United States,22750,1355618.0,52.4,87.18,59.59
Brasil,14576,865576.0,53.2,86.37,59.38
Spain,4021,237679.0,52.6,87.9,59.11
United Kingdom,4510,265870.0,52.7,86.57,58.95
Australia,2231,131440.0,53.2,84.36,58.92
France,4701,274526.0,51.9,86.49,58.4


In [16]:
# Rozrzut revenue per user w każdym wymiarze: max / min – porównanie z kanałami z A1 (~6%)

rpu = {by: segment_kpi(top_countries if by == "country" else customers, by)["revenue_per_user"]
       for by in ["traffic_source", "age_group", "gender", "country"]}

pd.Series({by: (s.max() / s.min() - 1) * 100 for by, s in rpu.items()}, name="spread_pct").round(1)

traffic_source     3.5
age_group          6.7
gender            13.1
country           12.3
Name: spread_pct, dtype: float64

## 4. Kanał pozyskania × segment

* Czy kanały przyciągają różnych klientów? Np. czy Facebook ma młodszych użytkowników niż Search?
* `pd.crosstab(..., normalize="index")` – udział grup wiekowych w każdym kanale (wiersz = 100%).

In [17]:
# Struktura wieku wg kanału – każdy wiersz = 100%

pd.crosstab(customers["traffic_source"], customers["age_group"], normalize="index").mul(100).round(1)

age_group,12-17,18-24,25-34,35-44,45-54,55-64,65+
traffic_source,,,,,,,
Display,10.1,12.3,17.2,16.5,17.0,16.9,10.1
Email,10.1,12.6,16.6,16.2,16.7,17.1,10.8
Facebook,10.3,11.8,16.5,17.8,17.2,16.2,10.2
Organic,10.2,12.0,16.9,16.8,16.7,16.8,10.5
Search,10.1,11.9,16.9,16.9,16.9,17.0,10.2


In [18]:
# Struktura płci wg kanału

pd.crosstab(customers["traffic_source"], customers["gender"], normalize="index").mul(100).round(1)

gender,F,M
traffic_source,,
Display,49.4,50.6
Email,49.4,50.6
Facebook,49.8,50.2
Organic,49.8,50.2
Search,49.9,50.1


## Wnioski (uzupełnij po uruchomieniu)

1. **Płeć to pierwszy wymiar z realną różnicą wartości:** revenue per user M 63,18 vs F 55,85 (+13%).
   Buyer rate jest równy (~52,8%), różnica wynika w całości z AOV (92,08 vs 81,27).
   Pokrywa się z różnicą cen produktów Men vs Women (+13%, A1) → przyczyną jest asortyment, nie zachowanie.
2. **Wiek, kraj i kanał nie różnicują wartości klienta.** Duże kraje (Chiny, USA, Brazylia) mają ~59,5;
   wyższe wartości Japonii i Niemiec to efekt małej próby. Skład demograficzny kanałów jest identyczny.
3. **Baza klientów nie rośnie:** stałe ~1 040 nowych użytkowników miesięcznie od 2019 r.
   Rynek skoncentrowany: Chiny, USA i Brazylia to 71% użytkowników.
Dwa ostrzeżenia z wyników (niegroźne)